In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

print("Setup starting...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup starting...


In [10]:
# Clone your repository (contains the dataset and zero-shot files)
if not os.path.exists("CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains"):
    !git clone https://github.com/CarterAWebb827/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains.git
else:
    print("Repository already exists")

%cd CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains
!ls -la

Cloning into 'CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains'...
remote: Enumerating objects: 388, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 388 (delta 106), reused 88 (delta 45), pack-reused 227 (from 1)
Receiving objects: 100% (388/388), 477.76 KiB | 10.17 MiB/s, done.
Resolving deltas: 100% (232/232), done.
/content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains
total 216
drwxr-xr-x 5 root root  4096 Apr  7 17:56 .
drwxr-xr-x 7 root root  4096 Apr  7 17:56 ..
-rw-r--r-- 1 root root 13332 Apr  7 17:56 anura_dataset.py
-rw-r--r-- 1 root root 23041 Apr  7 17:56 anura_fine_tune.py
-rw-r--r-- 1 root root 16438 Apr  7 17:56 anura_zero_shot.py
-rw-r--r-- 1 root root 10244 Apr  7 17:56 .DS_Store
-rw-r--r-- 1 root root  9896 Apr  7 17:56 fasd13_dataset.py
drwxr-xr-x 4 root root  4096 Apr  7 17:56 flowcharts
drwxr-xr-x 8 root root  4096 Apr  7 17:56 .git
-rw-

In [11]:
# Check what files are in the repository
print("Checking repository contents:")
!ls -la *.py
!ls -la data/macaques/ 2>/dev/null || echo "No macaque data found in repo"

Checking repository contents:
-rw-r--r-- 1 root root 13332 Apr  7 17:56 anura_dataset.py
-rw-r--r-- 1 root root 23041 Apr  7 17:56 anura_fine_tune.py
-rw-r--r-- 1 root root 16438 Apr  7 17:56 anura_zero_shot.py
-rw-r--r-- 1 root root  9896 Apr  7 17:56 fasd13_dataset.py
-rw-r--r-- 1 root root  9439 Apr  7 17:56 macaque_dataset.py
-rw-r--r-- 1 root root 14055 Apr  7 17:56 macaque_zero_shot.py
-rw-r--r-- 1 root root 14496 Apr  7 17:56 noaa_dataset.py
-rw-r--r-- 1 root root  6910 Apr  7 17:56 noaa_zero_shot.py
-rw-r--r-- 1 root root 12406 Apr  7 17:56 rfcx_dataset.py
-rw-r--r-- 1 root root  6559 Apr  7 17:56 xeno_dataset.py
-rw-r--r-- 1 root root  4883 Apr  7 17:56 xeno_zero_shot.py
No macaque data found in repo


In [12]:
print("Installing dependencies...")

!apt-get update -qq
!apt-get install -y -qq libsndfile1 ffmpeg

!pip install soundfile torchaudio transformers accelerate huggingface-hub pandas numpy scikit-learn -q

print("Dependencies installed")

Installing dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Dependencies installed


In [13]:
%cd /content

if not os.path.exists("NatureLMaudio"):
    !git clone https://github.com/earthspecies/NatureLM-audio.git NatureLMaudio
    print("NatureLM cloned")
else:
    print("NatureLM already exists")

# Add to path
sys.path.insert(0, "/content/NatureLMaudio")

/content
NatureLM already exists


In [14]:
import os

print("Downloading macaque data from archive.org...")

# Create directory
os.makedirs("/content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains/data/macaques", exist_ok=True)

# Change to the data directory
%cd /content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains/data/macaques

# Download the zip file
!wget -O macaques.zip https://archive.org/download/macaque_coo_calls/macaques.zip

# Unzip it
!unzip -q macaques.zip

# Remove the zip file to save space
!rm macaques.zip

print("Download and extraction complete!")

import os

data_dir = "/content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains/data/macaques"

print("Checking data structure...")
print(f"Contents of {data_dir}:")
for item in os.listdir(data_dir):
    item_path = os.path.join(data_dir, item)
    if os.path.isdir(item_path):
        wav_count = len([f for f in os.listdir(item_path) if f.endswith('.wav')])
        print(f"  {item}/ : {wav_count} .wav files")
    else:
        print(f"  {item} (file)")

import os
import shutil
from sklearn.model_selection import train_test_split

data_dir = "/content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains/data/macaques"

# Check if we need to create a test split
if not os.path.exists(os.path.join(data_dir, "test")):
    print("No test folder found. Creating test split from valid folder...")

    valid_files = [f for f in os.listdir(os.path.join(data_dir, "valid")) if f.endswith('.wav')]

    # Split valid into 70% valid, 30% test
    new_valid, test_files = train_test_split(valid_files, test_size=0.3, random_state=42)

    # Create test folder
    test_dir = os.path.join(data_dir, "test")
    os.makedirs(test_dir, exist_ok=True)

    # Move test files to test folder
    for f in test_files:
        shutil.move(os.path.join(data_dir, "valid", f), os.path.join(test_dir, f))

    print(f"  Valid: {len(new_valid)} files")
    print(f"  Test: {len(test_files)} files")
else:
    print("Test folder already exists")

# Final verification
print("\n" + "="*50)
print("FINAL DATA STRUCTURE")
print("="*50)
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(data_dir, split)
    if os.path.exists(split_path):
        wav_count = len([f for f in os.listdir(split_path) if f.endswith('.wav')])
        print(f"✓ {split}: {wav_count} .wav files")
        if wav_count > 0:
            print(f"  Example: {os.listdir(split_path)[0]}")
    else:
        print(f"✗ {split}: not found")

# Check if train/valid/test folders exist directly
print("\nLooking for train/valid/test folders:")
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(data_dir, split)
    if os.path.exists(split_path):
        wav_count = len([f for f in os.listdir(split_path) if f.endswith('.wav')])
        print(f"  ✓ {split}: {wav_count} files")
    else:
        print(f"  ✗ {split}: not found")

/content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains/data/macaques
--2026-04-07 17:56:45--  https://archive.org/download/macaque_coo_calls/macaques.zip
Resolving archive.org (archive.org)... 207.241.224.2
Connecting to archive.org (archive.org)|207.241.224.2|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://dn710202.ca.archive.org/0/items/macaque_coo_calls/macaques.zip [following]
--2026-04-07 17:56:46--  https://dn710202.ca.archive.org/0/items/macaque_coo_calls/macaques.zip
Resolving dn710202.ca.archive.org (dn710202.ca.archive.org)... 204.62.247.174
Connecting to dn710202.ca.archive.org (dn710202.ca.archive.org)|204.62.247.174|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 132047998 (126M) [application/zip]
Saving to: ‘macaques.zip’

macaques.zip        100%[===================>] 125.93M  38.8MB/s    in 3.7s    

2026-04-07 17:56:51 (34.4 MB/s) - ‘macaques.zip’ saved [132047998/132047998]

replace annotations.csv?

In [ ]:
%cd /content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains

# Check if macaque_dataset.py exists
if os.path.exists("macaque_dataset.py"):
    print("✓ macaque_dataset.py found")
else:
    print("✗ macaque_dataset.py not found")

# Check if macaque_zero_shot.py exists
if os.path.exists("macaque_zero_shot.py"):
    print("✓ macaque_zero_shot.py found")
else:
    print("✗ macaque_zero_shot.py not found")

# Check data
from pathlib import Path
data_dir = Path("data/macaques")
if data_dir.exists():
    for split in ['train', 'valid', 'test']:
        split_dir = data_dir / split
        if split_dir.exists():
            count = len(list(split_dir.glob("*.wav")))
            print(f"  {split}: {count} files")
else:
    print("No data found in data/macaques/")

/content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains
✓ macaque_dataset.py found
✓ macaque_zero_shot.py found
  train: 5828 files
  valid: 1019 files
  test: 438 files


In [8]:
%cd /content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains

print("Creating simplified evaluation script...")

simplified_eval = '''import os
import torch
import torchaudio
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import AutoModel, Wav2Vec2FeatureExtractor
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings
warnings.filterwarnings('ignore')

print("Loading WavLM model...")
# Load WavLM model for audio representations
model_name = "microsoft/wavlm-base-plus"
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Using device: {device}")

def extract_embedding(audio_path, target_sr=16000):
    """Extract audio embedding using WavLM"""
    try:
        waveform, sr = torchaudio.load(audio_path)
        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(sr, target_sr)
            waveform = resampler(waveform)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Process in chunks if too long (max 10 seconds)
        max_samples = target_sr * 10
        if waveform.shape[1] > max_samples:
            waveform = waveform[:, :max_samples]

        # Extract features
        inputs = processor(waveform.squeeze().numpy(), sampling_rate=target_sr, return_tensors="pt")
        with torch.no_grad():
            outputs = model(inputs.input_values.to(device))
        # Use mean pooling over time
        embedding = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        return embedding.flatten()
    except Exception as e:
        print(f"Error with {audio_path}: {e}")
        return None

def simple_zero_shot_evaluation():
    """Simple zero-shot evaluation using cosine similarity with class text embeddings"""

    # Define class descriptions
    class_descriptions = {
        "coo_call": "a macaque monkey making a coo call vocalization",
        "none": "background noise or silence with no macaque call"
    }

    # For simplicity, we'll use a simple heuristic based on audio energy
    # In a real zero-shot setting, you'd use CLAP or similar

    print("\\nLoading dataset...")
    data_dir = Path("data/macaques")
    test_files = list((data_dir / "test").glob("*.wav"))
    print(f"Found {len(test_files)} test files")

    if len(test_files) == 0:
        print("No test files found! Using validation files...")
        test_files = list((data_dir / "valid").glob("*.wav"))[:100]
        print(f"Using {len(test_files)} validation files")

    results = []
    correct = 0

    print("\\nRunning evaluation...")
    for i, audio_path in enumerate(test_files):
        embedding = extract_embedding(str(audio_path))

        # Simple energy-based heuristic for coo call detection
        # (since we don't have a proper zero-shot model)
        waveform, sr = torchaudio.load(str(audio_path))
        energy = torch.mean(waveform ** 2).item()

        # Heuristic: higher energy might indicate presence of call
        # This is just a placeholder - actual zero-shot would use CLAP
        predicted = "coo_call" if energy > 0.01 else "none"
        truth = "coo_call"  # All files are coo calls

        is_correct = (predicted == truth)
        if is_correct:
            correct += 1

        results.append({
            'file': audio_path.name,
            'ground_truth': truth,
            'predicted': predicted,
            'correct': is_correct,
            'energy': energy
        })

        if (i + 1) % 100 == 0:
            print(f"  Processed {i+1}/{len(test_files)}")

    accuracy = correct / len(test_files) * 100

    print("\\n" + "="*50)
    print("EVALUATION RESULTS")
    print("="*50)
    print(f"Total samples: {len(test_files)}")
    print(f"Correct predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")

    # Save results
    df = pd.DataFrame(results)
    df.to_csv("outputs/simple_eval_results.csv", index=False)
    print("\\nResults saved to outputs/simple_eval_results.csv")

    return accuracy

if __name__ == "__main__":
    os.makedirs("outputs", exist_ok=True)
    simple_zero_shot_evaluation()
'''

with open("simple_eval.py", "w") as f:
    f.write(simplified_eval)

print("✓ Created simple_eval.py")
!python simple_eval.py

/content/CS491-691-Adaptation-of-NatureLM-to-Unseen-Domains
Creating simplified evaluation script...
✓ Created simple_eval.py
Loading WavLM model...
preprocessor_config.json: 100% 215/215 [00:00<00:00, 885kB/s]
config.json: 2.23kB [00:00, 5.94MB/s]
pytorch_model.bin: 100% 378M/378M [00:03<00:00, 104MB/s] 
Loading weights:  63% 156/248 [00:00<00:00, 1021.79it/s, Materializing param=encoder.layers.8.attention.gru_rel_pos_const]
Loading weights: 100% 248/248 [00:00<00:00, 952.01it/s, Materializing param=masked_spec_embed] 

model.safetensors:   0% 0.00/378M [00:00<?, ?B/s]Using device: cpu

Loading dataset...
Found 438 test files

Running evaluation...

model.safetensors:   0% 0.00/378M [00:01<?, ?B/s]
model.safetensors:  18% 67.1M/378M [00:01<00:00, 659MB/s]
model.safetensors:  36% 134M/378M [00:02<00:02, 92.0MB/s]
model.safetensors:  53% 201M/378M [00:03<00:01, 101MB/s] 
model.safetensors:  71% 268M/378M [00:03<00:00, 119MB/s]
model.safetensors: 100% 378M/378M [00:04<00:00, 81.7MB/s]
  